In [2]:
import pandas as pd
import requests
import io
import os

# 1. Create temporary folders in Colab local storage
os.makedirs('/content/data', exist_ok=True)

print("⏳ Step 1: Downloading and processing SemEval-2016 Task 6 Data...")
semeval_url = "https://alt.qcri.org/semeval2016/task6/data/uploads/semeval2016-task6-trainingdata.txt"

# Download the file using requests
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(semeval_url, headers=headers)

# Parse SemEval (Usually ISO-8859-1 encoded, separated by tabs)
semeval_df = pd.read_csv(io.StringIO(response.text), sep='\t', encoding='latin1')

# Filter explicitly for Climate Change and standardise column names
# Columns are standardly: ID, Target, Tweet, Stance
semeval_cc = semeval_df[semeval_df['Target'] == 'Climate Change is a Real Concern'].copy()
semeval_cleaned = pd.DataFrame({
    'text': semeval_cc['Tweet'],
    'stance': semeval_cc['Stance'].str.capitalize() # Converts FAVOR -> Favor, AGAINST -> Against, NONE -> None
})
# Remap 'Favor' to match your task specification ('Favour')
semeval_cleaned['stance'] = semeval_cleaned['stance'].replace({'Favor': 'Favour'})

print(f"✅ Processed SemEval Climate Change subset. Rows: {len(semeval_cleaned)}")

print("\n⏳ Step 2: Processing uploaded GWSD.tsv file...")
# Read the local GWSD file you uploaded to Colab
gwsd_df = pd.read_csv('/content/GWSD.tsv', sep='\t')

# Calculate consensus stance by taking the statistical mode of the 8 annotators
worker_cols = [f'worker_{i}' for i in range(8)]
gwsd_df['consensus'] = gwsd_df[worker_cols].mode(axis=1)[0]

# Map the GWSD labels to your task classes
label_mapping = {
    'agrees': 'Favour',
    'disagrees': 'Against',
    'neutral': 'None'
}
gwsd_df['stance'] = gwsd_df['consensus'].map(label_mapping)

gwsd_cleaned = pd.DataFrame({
    'text': gwsd_df['sentence'],
    'stance': gwsd_df['stance']
})
print(f"✅ Processed GWSD dataset. Rows: {len(gwsd_cleaned)}")

print("\n⏳ Step 3: Merging datasets into unified alignment...")
# Combine both datasets into a single training source
unified_train = pd.concat([semeval_cleaned, gwsd_cleaned], ignore_index=True)

# Drop any potential missing values
unified_train.dropna(subset=['text', 'stance'], inplace=True)

# Save to your Colab local file space
output_path = '/content/data/final_english_train.csv'
unified_train.to_csv(output_path, index=False)

print(f"\n🎉 SUCCESS! Unified English training dataset saved to: {output_path}")
print("\n📊 Final Dataset Class Distribution:")
print(unified_train['stance'].value_counts())

⏳ Step 1: Downloading and processing SemEval-2016 Task 6 Data...
✅ Processed SemEval Climate Change subset. Rows: 395

⏳ Step 2: Processing uploaded GWSD.tsv file...
✅ Processed GWSD dataset. Rows: 2300

⏳ Step 3: Merging datasets into unified alignment...

🎉 SUCCESS! Unified English training dataset saved to: /content/data/final_english_train.csv

📊 Final Dataset Class Distribution:
stance
Favour     1151
None       1099
Against     445
Name: count, dtype: int64


In [3]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights based on your exact distribution
y_train = unified_train['stance'].values
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float)

print("Calculated Class Weights (Against will have the highest weight):", dict(zip(classes, weights)))

Calculated Class Weights (Against will have the highest weight): {'Against': np.float64(2.0187265917602994), 'Favour': np.float64(0.7804807413843035), 'None': np.float64(0.8174097664543525)}


In [4]:
import os
import pandas as pd

file_path = '/content/data/final_english_train.csv'

if os.path.exists(file_path):
    # 💡 The secret sauce: keep_default_na=False stops pandas from turning 'None' into NaN
    df = pd.read_csv(file_path, keep_default_na=False)

    print("--- Sample Favour Text ---")
    favour_samples = df[df['stance'] == 'Favour']['text']
    print(favour_samples.iloc[0] if not favour_samples.empty else "No Favour samples found")

    print("\n--- Sample Against Text ---")
    against_samples = df[df['stance'] == 'Against']['text']
    print(against_samples.iloc[0] if not against_samples.empty else "No Against samples found")

    print("\n--- Sample None Text ---")
    none_samples = df[df['stance'] == 'None']['text']
    print(none_samples.iloc[0] if not none_samples.empty else "No None samples found")

    # Check maximum length to make sure translation tokens won't truncate
    df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
    print(f"\n📊 Maximum word count in English text: {df['word_count'].max()}")
else:
    print("❌ Dataset file not found. Please rerun your initial download/setup cell!")

--- Sample Favour Text ---
We cant deny it, its really happening.  #SemST

--- Sample Against Text ---
The Climate Change people are disgusting assholes. Money transfer scheme for elite. May you rot.   #SemST

--- Sample None Text ---
#Netherlands just taught the rest of the world a very important lesson.  #SemST

📊 Maximum word count in English text: 60


In [5]:
import os
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report, f1_score

# 1. Load the dataset safely
file_path = '/content/data/final_english_train.csv'

if not os.path.exists(file_path):
    print("❌ Dataset not found! Please rerun the initial data setup cell.")
else:
    df = pd.read_csv(file_path, keep_default_na=False)

    # 2. Take a small test slice (e.g., 50 samples) to evaluate quickly without running out of memory
    # We sample evenly from each class to get an accurate evaluation
    sample_df = df.groupby('stance').sample(n=15, random_state=42).reset_index(drop=True)

    print("⏳ Loading Zero-Shot Classification Pipeline (BART-Large-MNLI)...")
    # This model runs completely free inside your Colab environment (uses CPU or GPU automatically)
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

    # 3. Define the explicit stance prompt mapping
    # Our candidate labels must represent our three classes
    candidate_labels = ["Favour", "Against", "None"]

    # Hypothesis template frames the problem explicitly around the target claim
    hypothesis_template = "This text expresses a stance that is {} toward the idea that climate change is a serious concern."

    print(f"⏳ Running zero-shot inference on {len(sample_df)} baseline rows...")

    predictions = []
    for text in sample_df['text']:
        result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template)
        # The top prediction label is the first element in the sorted results
        predictions.append(result['labels'][0])

    sample_df['predicted_stance'] = predictions

    # 4. Calculate performance using the track's target metric (Macro-F1)
    print("\n📊 --- BASELINE PERFORMANCE REPORT ---")
    macro_f1 = f1_score(sample_df['stance'], sample_df['predicted_stance'], average='macro')
    print(f"🎯 Baseline Macro-F1 Score: {macro_f1:.4f}\n")

    print(classification_report(sample_df['stance'], sample_df['predicted_stance']))

    # 5. Look at a few errors to analyze what the model missed
    print("❌ Sample Model Misclassifications:")
    errors = sample_df[sample_df['stance'] != sample_df['predicted_stance']]
    for i, row in errors.head(3).iterrows():
        print(f"\nText: {row['text']}")
        print(f"Actual: {row['stance']} | Predicted: {row['predicted_stance']}")

⏳ Loading Zero-Shot Classification Pipeline (BART-Large-MNLI)...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

⏳ Running zero-shot inference on 45 baseline rows...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



📊 --- BASELINE PERFORMANCE REPORT ---
🎯 Baseline Macro-F1 Score: 0.3556

              precision    recall  f1-score   support

     Against       0.53      0.53      0.53        15
      Favour       0.40      0.80      0.53        15
        None       0.00      0.00      0.00        15

    accuracy                           0.44        45
   macro avg       0.31      0.44      0.36        45
weighted avg       0.31      0.44      0.36        45

❌ Sample Model Misclassifications:

Text: Global warming is beneficial to the welfare and business.
Actual: Against | Predicted: Favour

Text: The climate is always changing in accordance with natural causes and recent changes.
Actual: Against | Predicted: Favour

Text: Data on how sea levels are rising, relied upon by the United Nations, is adjusted upward in “ arbitrary ” ways.
Actual: Against | Predicted: Favour


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [6]:
import os
import pandas as pd
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

# 1. Load data
file_path = '/content/data/final_english_train.csv'
df = pd.read_csv(file_path, keep_default_na=False)

# Let's take a small slice to test the translation environment first
test_slice = df.head(5).copy()

print("⏳ Loading M2M100 Translation Model (optimized for multilingual translation)...")
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"🚀 Model loaded on: {device.upper()}")

def translate_text(text, target_lang):
    # Set source language explicitly to English
    tokenizer.src_lang = "en"
    encoded_en = tokenizer(text, return_tensors="pt").to(device)

    # Generate tokens for target language
    # 'hi' for Hindi, 'bn' for Bengali
    generated_tokens = model.generate(**encoded_en, forced_bos_token_id=tokenizer.get_lang_id(target_lang))
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

print("\n⏳ Testing translation on 5 sample rows...")
print("--- Hindi Translation Test ---")
test_slice['text_hi'] = test_slice['text'].apply(lambda x: translate_text(x, "hi"))
print(test_slice[['text', 'text_hi']].head(2))

print("\n--- Bengali Translation Test ---")
test_slice['text_bn'] = test_slice['text'].apply(lambda x: translate_text(x, "bn"))
print(test_slice[['text', 'text_bn']].head(2))

print("\n🎉 Translation environment verified successfully!")

⏳ Loading M2M100 Translation Model (optimized for multilingual translation)...


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

🚀 Model loaded on: CUDA

⏳ Testing translation on 5 sample rows...
--- Hindi Translation Test ---
                                                text  \
0     We cant deny it, its really happening.  #SemST   
1  RT @cderworiz: Timelines are short. Strategy m...   

                                             text_hi  
0  हम इसे अस्वीकार कर सकते हैं, यह वास्तव में हो ...  
1  RT @cderworiz: समय सीमा छोटी है. रणनीति को दिस...  

--- Bengali Translation Test ---
                                                text  \
0     We cant deny it, its really happening.  #SemST   
1  RT @cderworiz: Timelines are short. Strategy m...   

                                             text_bn  
0         আমরা এটা ভুল করি, এটি সত্যিই ঘটেছে. #SemST  
1  RT @cderworiz: টাইম লাইন কঠিন. কৌশলটি ডিসেম্বর...  

🎉 Translation environment verified successfully!


In [7]:
import os
import pandas as pd
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from tqdm import tqdm

# 1. Double check the hardware acceleration status
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Environment status: Model will load on {device.upper()}")
if device == "cpu":
    print("❌ CRITICAL: You are still on CPU. Please go to Runtime > Change runtime type and select T4 GPU!")

# 2. Reload the cleaned base dataset from storage
file_path = '/content/data/final_english_train.csv'
if not os.path.exists(file_path):
    print("❌ Dataset missing from current session memory. Please rerun your initial download/setup cell first.")
else:
    df = pd.read_csv(file_path, keep_default_na=False)

    # 3. Initialize model components
    model_name = "facebook/m2m100_418M"
    tokenizer = M2M100Tokenizer.from_pretrained(model_name)
    model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)

    # Batched translation accelerator function
    def batch_translate(texts, target_lang, batch_size=32):
        translated_texts = []
        tokenizer.src_lang = "en"

        for i in tqdm(range(0, len(texts), batch_size), desc=f"Translating to {target_lang}"):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
            generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id(target_lang))
            decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            translated_texts.extend(decoded)

        return translated_texts

    # 4. Execute rapid complete translation runs
    texts_list = df['text'].tolist()
    df['text_hi'] = batch_translate(texts_list, "hi", batch_size=32)
    df['text_bn'] = batch_translate(texts_list, "bn", batch_size=32)

    # 5. Save the final target language dataset back to local storage
    output_path = '/content/data/translated_train.csv'
    df.to_csv(output_path, index=False)
    print(f"\n🎉 Success! Entire cross-lingual training engine built and saved to: {output_path}")

🚀 Environment status: Model will load on CUDA


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Translating to bn: 100%|██████████| 85/85 [24:30<00:00, 17.30s/it]


🎉 Success! Entire cross-lingual training engine built and saved to: /content/data/translated_train.csv


In [12]:
import os
import pandas as pd
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer, pipeline
from tqdm import tqdm

# =====================================================================
# 1. DIRECT EXCEL LOADING (Matches your exact sidebar files!)
# =====================================================================
bn_path = '/content/Bangla_500_test_data.xlsx'
hi_path = '/content/Hindi_500_test_data.xlsx'
output_dir = '/content/outputs'
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(bn_path) or not os.path.exists(hi_path):
    print("❌ Error: Files not found. Verify they are exactly named in the sidebar.")
else:
    print("✅ Excel files found successfully! Loading data...")
    # Read the excel sheets directly
    df_bn = pd.read_excel(bn_path)
    df_hi = pd.read_excel(hi_path)

    # =====================================================================
    # 2. RUNTIME AND HARDWARE CHECK
    # =====================================================================
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🚀 System verified. Models running on: {device.upper()}")

    # =====================================================================
    # 3. TRANSLATION TO ENGLISH (BATCH RUNS)
    # =====================================================================
    print("\n🌐 Step 1: Loading M2M100 Translation Model...")
    model_name = "facebook/m2m100_418M"
    trans_tokenizer = M2M100Tokenizer.from_pretrained(model_name)
    trans_model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)

    def translate_to_english_batched(texts, src_lang, batch_size=32):
        translated = []
        trans_tokenizer.src_lang = src_lang

        for i in tqdm(range(0, len(texts), batch_size), desc=f"Translating {src_lang.upper()} to English"):
            batch = [str(t) for t in texts[i:i+batch_size]]
            encoded = trans_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            generated = trans_model.generate(**encoded, forced_bos_token_id=trans_tokenizer.get_lang_id("en"))
            decoded = trans_tokenizer.batch_decode(generated, skip_special_tokens=True)
            translated.extend(decoded)
        return translated

    print("\nExecuting translation batch passes...")
    df_bn['Eng_Translation'] = translate_to_english_batched(df_bn['COMMENT'].tolist(), "bn", batch_size=32)
    df_hi['Eng_Translation'] = translate_to_english_batched(df_hi['COMMENT'].tolist(), "hi", batch_size=32)

    # =====================================================================
    # 4. ZERO-SHOT STANCE SELECTION
    # =====================================================================
    print("\n⏳ Step 2: Activating Zero-Shot Classifier Model...")
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if device == "cuda" else -1)

    candidate_labels = ["Favour", "Against", "None"]
    hypothesis = "This text expresses a stance that is {} toward the idea that climate change is a serious concern."

    def predict_batch_stance(texts, batch_size=32):
        preds = []
        for out in tqdm(classifier(texts, candidate_labels, hypothesis_template=hypothesis, batch_size=batch_size), desc="Classifying Stance"):
            preds.append(out['labels'][0])
        return preds

    print("\n🎯 Step 3: Resolving stance predictions from translations...")
    df_bn['Prediction'] = predict_batch_stance(df_bn['Eng_Translation'].tolist(), batch_size=32)
    df_hi['Prediction'] = predict_batch_stance(df_hi['Eng_Translation'].tolist(), batch_size=32)

    # =====================================================================
    # 5. SUBMISSION PREPARATION & SAVE
    # =====================================================================
    print("\n💾 Step 4: Formatting and saving official submission file...")

    run_1_sub = pd.concat([
        pd.DataFrame({'ID': df_bn['ID'], 'Prediction': df_bn['Prediction']}),
        pd.DataFrame({'ID': df_hi['ID'], 'Prediction': df_hi['Prediction']})
    ], ignore_index=True)

    submission_path = f"{output_dir}/run1_translate_test_submission.csv"
    run_1_sub.to_csv(submission_path, index=False)

    print(f"\n🎉 RUN 1 COMPLETELY SECURED!")
    print(f"💾 File successfully built at: {submission_path}")
    print("\n📊 Generated Submission Label Distribution:")
    print(run_1_sub['Prediction'].value_counts())

✅ Excel files found successfully! Loading data...
🚀 System verified. Models running on: CUDA

🌐 Step 1: Loading M2M100 Translation Model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]


Executing translation batch passes...


Translating HI to English: 100%|██████████| 16/16 [07:26<00:00, 27.88s/it]



⏳ Step 2: Activating Zero-Shot Classifier Model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


🎯 Step 3: Resolving stance predictions from translations...


Classifying Stance: 100%|██████████| 500/500 [00:00<00:00, 931239.79it/s]


💾 Step 4: Formatting and saving official submission file...

🎉 RUN 1 COMPLETELY SECURED!
💾 File successfully built at: /content/outputs/run1_translate_test_submission.csv

📊 Generated Submission Label Distribution:
Prediction
Favour     539
Against    451
None        10
Name: count, dtype: int64


In [13]:
from google.colab import files
files.download('/content/outputs/run1_translate_test_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer
from datasets import Dataset

# 1. Load the cross-lingual training engine we built earlier
train_path = '/content/data/translated_train.csv'

if not os.path.exists(train_path):
    print("❌ Dataset missing from current session. Please ensure translated_train.csv is in the data folder.")
else:
    # Read the data, ensuring 'None' strings aren't treated as empty NaNs
    df_train = pd.read_csv(train_path, keep_default_na=False)

    print("⏳ Initializing MuRIL Tokenizer Setup...")
    model_ckpt = "google/muril-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

    # Map text classes into numeric IDs for cross-entropy optimization
    label_map = {"Favour": 0, "Against": 1, "None": 2}

    # We create two distinct sub-datasets: one for Hindi targets, one for Bengali targets
    # This prepares your architecture for native fine-tuning
    train_hi = pd.DataFrame({'text': df_train['text_hi'], 'label': df_train['stance'].map(label_map)})
    train_bn = pd.DataFrame({'text': df_train['text_bn'], 'label': df_train['stance'].map(label_map)})

    # Preview to verify alignment
    print("\n📝 Sample Aligned Hindi Token Feed:")
    print(train_hi.head(2))

    print("\n🎉 Ready to initiate deep learning fine-tuning sequence!")

⏳ Initializing MuRIL Tokenizer Setup...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]


📝 Sample Aligned Hindi Token Feed:
                                                text  label
0  हम इसे अस्वीकार कर सकते हैं, यह वास्तव में हो ...      0
1  RT @cderworiz: समय सीमा छोटी है. रणनीति को दिस...      0

🎉 Ready to initiate deep learning fine-tuning sequence!


In [16]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

# =====================================================================
# 1. DATA AGGREGATION & ALIGNMENT
# =====================================================================
df_train = pd.read_csv('/content/data/translated_train.csv', keep_default_na=False)
label_map = {"Favour": 0, "Against": 1, "None": 2}

# Construct language data nodes
hi_data = pd.DataFrame({'text': df_train['text_hi'], 'label': df_train['stance'].map(label_map)})
bn_data = pd.DataFrame({'text': df_train['text_bn'], 'label': df_train['stance'].map(label_map)})

# Blend both languages to maximize cross-lingual boundary representation
blended_df = pd.concat([hi_data, bn_data], ignore_index=True).dropna().reset_index(drop=True)

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(blended_df)
# Split into 90% Train and 10% Local Validation to monitor overfitting
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

# =====================================================================
# 2. TOKENIZATION PASSTHROUGH
# =====================================================================
model_ckpt = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_datasets = dataset_split.map(tokenize_function, batched=True)

# =====================================================================
# 3. COMBATING IMBALANCE WITH CUSTOM LOSS WEIGHTS
# =====================================================================
y_train = blended_df['label'].values
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")

class ImbalanceWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Direct class-weighted Cross-Entropy Loss to protect the minority class ('Against')
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# =====================================================================
# 4. MODEL ARCHITECTURE & HYPERPARAMETERS
# =====================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    # Target Metric validation evaluation
    macro_f1 = f1_score(labels, preds, average='macro')
    return {"macro_f1": macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)

training_args = TrainingArguments(
    output_dir="./muril_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3, # 3 epochs balances stable convergence without local overfitting
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True, # Mixed precision scaling speeds up T4 GPU calculations
    logging_steps=50,
    report_to="none"
)

trainer = ImbalanceWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

print("\n⏳ Commencing MuRIL Deep Learning Fine-Tuning Epochs...")
trainer.train()
print("🎉 Model Fine-Tuning Complete! Best Model Checkpoint Secured.")

Map:   0%|          | 0/4851 [00:00<?, ? examples/s]

Map:   0%|          | 0/539 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w


⏳ Commencing MuRIL Deep Learning Fine-Tuning Epochs...


Epoch,Training Loss,Validation Loss,Macro F1
1,1.047509,1.010959,0.611598
2,0.952008,0.921514,0.650991
3,0.873366,0.889821,0.664180


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Model Fine-Tuning Complete! Best Model Checkpoint Secured.


In [17]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# 1. Verify files are still in local storage
bn_path = '/content/Bangla_500_test_data.xlsx'
hi_path = '/content/Hindi_500_test_data.xlsx'
output_dir = '/content/outputs'

if not os.path.exists(bn_path) or not os.path.exists(hi_path):
    print("❌ Error: Missing test files. Please re-upload them to your sidebar panel.")
else:
    print("✅ Test sets verified. Preparing MuRIL Inference Pipeline...")
    df_bn = pd.read_excel(bn_path)
    df_hi = pd.read_excel(hi_path)

    # 2. Hardware Allocation
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Inverse map to turn numeric predictions back into string tags for submission
    id_to_label = {0: "Favour", 1: "Against", 2: "None"}

    # 3. Prediction Helper Function
    def predict_muril_stance(texts, batch_size=32):
        model.eval()  # Put model into evaluation mode
        all_preds = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc="Predicting Stance"):
                batch = [str(t) for t in texts[i:i+batch_size]]

                # Tokenize batch exactly like training setup
                inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

                # Forward pass to get logits
                outputs = model(**inputs)
                logits = outputs.logits.cpu().numpy()

                # Select index of highest probability value
                batch_preds = np.argmax(logits, axis=1)
                all_preds.extend([id_to_label[p] for p in batch_preds])

        return all_preds

    # 4. Generate Predictions Directly on the Native Hindi and Bangla Comments
    print("\n🇮🇳 Running inference on Hindi test comments...")
    df_hi['Prediction'] = predict_muril_stance(df_hi['COMMENT'].tolist(), batch_size=32)

    print("\n🇧🇩 Running inference on Bangla test comments...")
    df_bn['Prediction'] = predict_muril_stance(df_bn['COMMENT'].tolist(), batch_size=32)

    # 5. Format to Official Competition Layout
    run_2_sub = pd.concat([
        pd.DataFrame({'ID': df_bn['ID'], 'Prediction': df_bn['Prediction']}),
        pd.DataFrame({'ID': df_hi['ID'], 'Prediction': df_hi['Prediction']})
    ], ignore_index=True)

    submission_path = f"{output_dir}/run2_muril_fine_tuned_submission.csv"
    run_2_sub.to_csv(submission_path, index=False)

    print(f"\n🎉 RUN 2 COMPLETELY SECURED!")
    print(f"💾 File built at: {submission_path}")
    print("\n📊 Run 2 Submission Label Distribution:")
    print(run_2_sub['Prediction'].value_counts())

✅ Test sets verified. Preparing MuRIL Inference Pipeline...

🇮🇳 Running inference on Hindi test comments...


Predicting Stance: 100%|██████████| 16/16 [00:02<00:00,  7.97it/s]



🇧🇩 Running inference on Bangla test comments...


Predicting Stance: 100%|██████████| 16/16 [00:01<00:00, 11.01it/s]


🎉 RUN 2 COMPLETELY SECURED!
💾 File built at: /content/outputs/run2_muril_fine_tuned_submission.csv

📊 Run 2 Submission Label Distribution:
Prediction
Favour     418
None       343
Against    239
Name: count, dtype: int64


In [18]:
from google.colab import files
files.download('/content/outputs/run2_muril_fine_tuned_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

# 1. Pipeline Data Setup
df_train = pd.read_csv('/content/data/translated_train.csv', keep_default_na=False)
label_map = {"Favour": 0, "Against": 1, "None": 2}

hi_data = pd.DataFrame({'text': df_train['text_hi'], 'label': df_train['stance'].map(label_map)})
bn_data = pd.DataFrame({'text': df_train['text_bn'], 'label': df_train['stance'].map(label_map)})
blended_df = pd.concat([hi_data, bn_data], ignore_index=True).dropna().reset_index(drop=True)

dataset = Dataset.from_pandas(blended_df).train_test_split(test_size=0.1, seed=42)

# =====================================================================
# 2. XLM-ROBERTA SETUP (Open, Non-Gated Alternative)
# =====================================================================
print("⏳ Initializing XLM-RoBERTa Tokenizer Workspace...")
model_ckpt = "xlm-roberta-base"  # Completely open model checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. Class Weight Balancing System
y_train = blended_df['label'].values
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")

class XLMRobertaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)

training_args = TrainingArguments(
    output_dir="./xlmr_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average='macro')}

trainer = XLMRobertaWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

print("\n⏳ Fine-tuning XLM-RoBERTa Architecture...")
trainer.train()

# =====================================================================
# 4. RUNNING TEST INFERENCE FOR RUN 3
# =====================================================================
df_bn = pd.read_excel('/content/Bangla_500_test_data.xlsx')
df_hi = pd.read_excel('/content/Hindi_500_test_data.xlsx')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
id_to_label = {0: "Favour", 1: "Against", 2: "None"}

def predict_stance(texts, batch_size=32):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
            batch = [str(t) for t in texts[i:i+batch_size]]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            batch_preds = np.argmax(outputs.logits.cpu().numpy(), axis=1)
            all_preds.extend([id_to_label[p] for p in batch_preds])
    return all_preds

print("\n🎯 Gathering final Run 3 stance tags...")
df_hi['Prediction'] = predict_stance(df_hi['COMMENT'].tolist())
df_bn['Prediction'] = predict_stance(df_bn['COMMENT'].tolist())

run_3_sub = pd.concat([
    pd.DataFrame({'ID': df_bn['ID'], 'Prediction': df_bn['Prediction']}),
    pd.DataFrame({'ID': df_hi['ID'], 'Prediction': df_hi['Prediction']})
], ignore_index=True)

run_3_sub.to_csv('/content/outputs/run3_xlm_roberta_submission.csv', index=False)
print("\n🎉 RUN 3 COMPLETELY SECURED AND SAVED!")
print(run_3_sub['Prediction'].value_counts())

⏳ Initializing XLM-RoBERTa Tokenizer Workspace...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎯 Gathering final Run 3 stance tags...


Predicting: 100%|██████████| 16/16 [00:00<00:00, 16.63it/s]


🎉 RUN 3 COMPLETELY SECURED AND SAVED!
Prediction
Against    387
Favour     329
None       284
Name: count, dtype: int64


In [21]:
from google.colab import files
files.download('/content/outputs/run3_xlm_roberta_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>